In [4]:
# ============================================================
# 02_mouse_features.ipynb
# Adaptive Continuous Authentication — Mouse Feature Builder
# ============================================================

"""
Goal:
  - Load mouse_raw.parquet (Balabit events)
  - For each (user_id, session_id), compute behavioral features:
      * velocity / acceleration stats
      * curvature / direction-change stats
      * path length & session duration
      * idle ratio (time gaps)
      * event-type proportions
  - Save compact feature arrays for modeling

Output:
  data/mouse_features.npz
"""

# ============================================================
# Setup
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
import scipy.stats as st

DATA_DIR = Path("data")
RAW_PATH = DATA_DIR / "mouse_raw.parquet"
OUT_PATH = DATA_DIR / "mouse_features.npz"

print("📥 Loading raw mouse events...")
df = pd.read_parquet(RAW_PATH, engine="fastparquet")
print(f"Loaded {len(df)} events for {df['user_id'].nunique()} users.")

# ============================================================
# 1. Helper functions
# ============================================================

def stats_1d(x):
    x = np.asarray(x, dtype=np.float32)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return {
            "mean": 0.0, "std": 0.0, "median": 0.0, "iqr": 0.0,
            "skew": 0.0, "kurt": 0.0, "min": 0.0, "max": 0.0
        }
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x)),
        "median": float(np.median(x)),
        "iqr": float(st.iqr(x)),
        "skew": float(st.skew(x)),
        "kurt": float(st.kurtosis(x)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
    }

def entropy_measure(x, num_bins=20):
    x = np.asarray(x, dtype=np.float32)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return 0.0
    hist, _ = np.histogram(x, bins=num_bins, density=True)
    hist = hist + 1e-8
    return float(-np.sum(hist * np.log(hist)))

# ============================================================
# 2. Session-level feature extraction
# ============================================================

# We'll extract per (user_id, session_id)
group_cols = ["user_id", "session_id"]

rows = []
user_ids = []
session_ids = []

IDLE_THRESHOLD = 1.0  # seconds

print("🔧 Extracting session-level features...")
for (uid, sid), g in df.groupby(group_cols):
    g = g.sort_values("timestamp")
    t = g["timestamp"].to_numpy(dtype=np.float32)
    x = g["x"].to_numpy(dtype=np.float32)
    y = g["y"].to_numpy(dtype=np.float32)
    states = g["state"].astype(str).to_numpy()

    if len(t) < 3:
        # Too short for meaningful dynamics
        continue

    dt = np.diff(t)
    dx = np.diff(x)
    dy = np.diff(y)
    dist = np.sqrt(dx**2 + dy**2)

    # Prevent tiny or zero dt (causes huge velocities)
    dt_safe = np.clip(dt, 1e-3, None)        # minimum 1ms
    
    # Distance magnitude
    dist = np.sqrt(np.clip(dx*dx + dy*dy, 0, 1e12))  # avoid overflow
    
    vel = dist / dt_safe
    
    # Clip velocity to sane max (10000 px/sec is plenty)
    vel = np.clip(vel, 0, 1e4)
    
    # Acceleration: diff(vel)/dt — also stabilize dt
    acc_dt = np.clip(dt_safe[1:], 1e-3, None)
    acc_raw = np.diff(vel) / acc_dt
    
    # Clip acceleration to prevent blowups
    acc = np.clip(acc_raw, -1e4, 1e4)

    # Direction & curvature
    directions = np.arctan2(dy, dx)
    dtheta = np.diff(directions)
    curvature = np.abs(np.unwrap(dtheta))

    # Session duration & path length
    duration = float(t[-1] - t[0])
    path_length = float(dist.sum())

    # Idle ratio: fraction of large gaps
    idle_ratio = float(np.mean(dt > IDLE_THRESHOLD))

    # Movement density (events per second)
    events_per_sec = float(len(t) / (duration + 1e-4))

    # Event-type proportions
    # (state strings vary like "Move", "Press", "Release", etc. in Balabit)
    states_series = pd.Series(states)
    total_events = len(states)
    move_ratio = float((states_series == "Move").mean())
    press_ratio = float((states_series == "Press").mean()) if "Press" in states_series.values else 0.0
    release_ratio = float((states_series == "Release").mean()) if "Release" in states_series.values else 0.0

    # Stats on vel, acc, curvature
    vel_stats = stats_1d(vel)
    acc_stats = stats_1d(acc)
    curv_stats = stats_1d(curvature)

    vel_entropy = entropy_measure(vel)
    curv_entropy = entropy_measure(curvature)

    # Flatten features into vector
    feats = [
        # Velocity stats (8)
        vel_stats["mean"], vel_stats["std"], vel_stats["median"], vel_stats["iqr"],
        vel_stats["skew"], vel_stats["kurt"], vel_stats["min"], vel_stats["max"],

        # Acceleration stats (8)
        acc_stats["mean"], acc_stats["std"], acc_stats["median"], acc_stats["iqr"],
        acc_stats["skew"], acc_stats["kurt"], acc_stats["min"], acc_stats["max"],

        # Curvature stats (8)
        curv_stats["mean"], curv_stats["std"], curv_stats["median"], curv_stats["iqr"],
        curv_stats["skew"], curv_stats["kurt"], curv_stats["min"], curv_stats["max"],

        # Behavioral scalars (7)
        duration,
        path_length,
        idle_ratio,
        events_per_sec,
        move_ratio,
        press_ratio,
        release_ratio,

        # Entropies (2)
        vel_entropy,
        curv_entropy,
    ]

    rows.append(feats)
    user_ids.append(uid)
    session_ids.append(sid)

X = np.array(rows, dtype=np.float32)
user_ids = np.array(user_ids, dtype=np.int32)
session_ids = np.array(session_ids, dtype=object)

print(f"\nSession-feature matrix shape: {X.shape}")
print("Unique users:", np.unique(user_ids).shape[0])

# ============================================================
# 3. Save output
# ============================================================

print(f"💾 Saving mouse features → {OUT_PATH}")
np.savez_compressed(
    OUT_PATH,
    features=X,
    user_id=user_ids,
    session_id=session_ids,
)

print("✅ Done. Mouse features ready for encoder training.")

📥 Loading raw mouse events...
Loaded 4609929 events for 10 users.
🔧 Extracting session-level features...

Session-feature matrix shape: (1676, 33)
Unique users: 10
💾 Saving mouse features → data/mouse_features.npz
✅ Done. Mouse features ready for encoder training.


In [3]:
df = pd.read_parquet("data/mouse_raw.parquet")

print("User count:", df["user_id"].nunique())
print("Total events:", len(df))

print("\nSessions per user:")
print(df.groupby("user_id")["session_id"].nunique())

User count: 10
Total events: 2253816

Sessions per user:
user_id
7     7
9     7
12    7
15    6
16    6
20    7
21    7
23    6
29    7
35    5
Name: session_id, dtype: int64
